In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [2]:
#path = Path("/content/Loan-Approval-Prediction.csv")

In [3]:
data = pd.read_csv("Loan-Approval-Prediction.csv")

In [ ]:
data.head(10)

In [ ]:
data.info()

In [ ]:
data.describe().T

In [ ]:
data.shape

In [ ]:
data.isnull().sum()

In [ ]:
data.duplicated()

In [10]:
data.columns = data.columns.str.strip().str.lower().str.replace(" ", "_")

In [ ]:
data.head()

In [12]:
data = data.rename(columns={'applicantincome': 'applicant_income', 'coapplicantincome': 'co_applicant_income', 'loanamount': 'loan_amount'})

In [ ]:
data.head(10)

In [14]:
data.drop(['loan_id'], axis=1, inplace=True)

In [ ]:
data.head()

In [ ]:
missing_categorical_cols = ['gender', 'married', 'dependents', 'self_employed']
for col in missing_categorical_cols:
  data[col].fillna(data[col].mode()[0], inplace=True)

In [ ]:
missing_numerical_cols = ['loan_amount_term', 'loan_amount', 'credit_history']
for col in missing_numerical_cols:
  data[col].fillna(data[col].median(), inplace=True)

In [ ]:
data.isnull().sum()

In [ ]:
data.info()

In [20]:
data.to_csv('cleaned_dataset.csv', index=False)

In [21]:
data.to_csv('cleaned_dataset.csv', index=False)

In [ ]:
import seaborn as sns
sns.countplot(x='loan_status', data=data)
plt.xlabel("Loan Status")
plt.ylabel("Count")
plt.title("Loan Approval Distribution")
plt.show()

In [ ]:
sns.countplot(x='credit_history',hue='loan_status',data=data)
plt.xlabel("Credit History")
plt.ylabel("Count")
plt.title("Credit History vs Loan Status")
plt.show()

In [ ]:
sns.countplot(
    x='education',
    hue='loan_status',
    data=data
)
plt.xlabel("Education")
plt.ylabel("Count")
plt.title("Educational Status vs Loan Status")
plt.show()

In [ ]:
sns.countplot(
    x='gender',
    hue='loan_status',
    data=data
)
plt.xlabel("Gender")
plt.ylabel("Count")
plt.title("Gender vs Loan Status")
plt.show()

In [ ]:
sns.countplot(
    x='married',
    hue='loan_status',
    data=data
)
plt.xlabel("Married")
plt.ylabel("Count")
plt.title("Marital Status vs Loan Status")
plt.show()

In [ ]:
sns.countplot(
    x='self_employed',
    hue='loan_status',
    data=data
)
plt.xlabel("Self Employed")
plt.ylabel("Count")
plt.title("Employment Status vs Loan Status")
plt.show()

In [ ]:
sns.countplot(
    x='property_area',
    hue='loan_status',
    data=data
)
plt.xlabel("Property")
plt.ylabel("Count")
plt.title("Property Owner vs Loan Status")
plt.show()

In [ ]:
sns.countplot(
    x='property_area',
    hue='credit_history',
    data=data
)
plt.xlabel("Property")
plt.ylabel("Count")
plt.title("Property Owner vs Credit History")
plt.show()

In [ ]:
sns.countplot(
    x='married',
    hue='self_employed',
    data=data
)
plt.xlabel("Marital Status")
plt.ylabel("Count")
plt.title("Marital Status vs Credit History")
plt.show()

In [ ]:
sns.histplot(data['applicant_income'], kde=True)

In [ ]:
sns.histplot(data['loan_amount'], kde=True)
plt.show()

In [33]:
data['total_income'] = data['applicant_income'] + data['co_applicant_income']

In [ ]:
sns.histplot(data['total_income'], kde=True)

In [35]:
data['income_loan_ratio'] = (data['total_income'] / data['loan_amount'])

In [36]:
data['total_income_log'] = np.log(data['total_income'])

data['loan_amount_log'] = np.log(data['loan_amount'])

In [ ]:
sns.histplot(data['total_income_log'], kde=True)

In [38]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

categorical_columns = ['gender','married','education','self_employed',
                       'property_area','loan_status','dependents']

for col in categorical_columns:
    data[col] = le.fit_transform(data[col])

In [39]:
X = data.drop(['loan_status'], axis=1)
y = data['loan_status']

In [40]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [41]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [42]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
lr = LogisticRegression()
lr.fit(X_train_scaled, y_train)
lr_pred = lr.predict(X_test_scaled)

In [ ]:
from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier()
dt.fit(X_train_scaled, y_train)
dt_pred = dt.predict(X_test_scaled)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier()
rf.fit(X_train_scaled, y_train)
rf_pred = rf.predict(X_test_scaled)

In [ ]:
print("Logistic Regression:",
      accuracy_score(y_test, lr_pred))

print("Decision Tree:",
      accuracy_score(y_test, dt_pred))

print("Random Forest:",
      accuracy_score(y_test, rf_pred))

In [ ]:
cm = confusion_matrix(y_test, rf_pred)

plt.figure(figsize=(6,4))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Rejected', 'Approved'],
    yticklabels=['Rejected', 'Approved']
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - Random Forest")

plt.show()

In [ ]:
report = classification_report(
    y_test,
    rf_pred,
    output_dict=True
)

report_df = pd.DataFrame(report).transpose()

report_df

In [ ]:
importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf.feature_importances_
})

importance.sort_values(
    by='Importance',
    ascending=False,
    inplace=True
)

importance

In [ ]:
import pickle

pickle.dump(rf, open("loan_approval_model.pkl", "wb"))